In [ ]:
#USED
# requirements:
# pip install pillow tqdm scikit-image opencv-python --quiet

from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from scipy.signal import wiener
from skimage.morphology import disk, white_tophat
from skimage import exposure
import cv2
import os

# ---------------------------
# USER CONFIG
# ---------------------------
INPUT_DIR  = Path("my_unzipped_dataset")   # change to your input folder
OUTPUT_DIR = Path("enhanced_dataset_all")  # root output folder
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

# Step subfolders (single folder per step)
STEP0_DIR = OUTPUT_DIR / "step0_foreground_rgba"      # raw RGBA from GrabCut
STEP1_DIR = OUTPUT_DIR / "fg_step1_gray"
STEP2_DIR = OUTPUT_DIR / "fg_step2_wiener"
STEP3_DIR = OUTPUT_DIR / "fg_step3_tophat"
STEP4_DIR = OUTPUT_DIR / "fg_step4_stretch"
STEP5_DIR = OUTPUT_DIR / "fg_step5_CLAHE"
STEP_FINAL_DIR = OUTPUT_DIR / "fg_final_rgba"         # final RGBA with processed grayscale as RGB

for d in (STEP0_DIR, STEP1_DIR, STEP2_DIR, STEP3_DIR, STEP4_DIR, STEP5_DIR, STEP_FINAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------
# HELPERS
# ---------------------------
def grayscale_custom(img_np):
    """Convert RGB numpy array to custom grayscale (uint8)."""
    R = img_np[..., 0].astype(float)
    G = img_np[..., 1].astype(float)
    B = img_np[..., 2].astype(float)
    gray = 0.2989 * R + 0.5870 * G + 0.1140 * B
    return np.clip(gray, 0, 255).astype(np.uint8)

def safe_name_from_path(p: Path, base: Path):
    rel = p.relative_to(base)
    rel_str = str(rel)
    safe = rel_str.replace(os.sep, "__").replace("/", "__").replace(" ", "_")
    return safe

def extract_foreground_grabcut(img_path, iter_count=5, rect_margin=10):
    """
    Use GrabCut to extract foreground.
    Returns:
      rgba: (H,W,4) uint8 with alpha 0 background, 255 foreground
      mask_fg: (H,W) uint8 mask where foreground==1, background==0
    """
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        raise RuntimeError(f"cv2.imread failed for {img_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    mask = np.zeros((h, w), np.uint8)
    bgdModel = np.zeros((1, 65), np.float64)
    fgdModel = np.zeros((1, 65), np.float64)

    x = rect_margin
    y = rect_margin
    rect_w = max(1, w - 2 * rect_margin)
    rect_h = max(1, h - 2 * rect_margin)
    rect = (x, y, rect_w, rect_h)

    cv2.grabCut(img_rgb, mask, rect, bgdModel, fgdModel, iter_count, cv2.GC_INIT_WITH_RECT)

    # foreground mask: sure or probable foreground
    mask_fg = np.where((mask == 1) | (mask == 3), 1, 0).astype('uint8')

    # apply mask to RGB image (background becomes 0)
    fg_rgb = img_rgb * mask_fg[:, :, np.newaxis]

    alpha = (mask_fg * 255).astype(np.uint8)
    rgba = np.dstack((fg_rgb, alpha))

    return rgba, mask_fg

def enhance_pipeline_on_foreground(rgb_fg, mask_fg):
    """
    rgb_fg: uint8 RGB image where background pixels are zero
    mask_fg: uint8 mask (1 foreground, 0 background)
    Returns: tuple of uint8 images (gray, wiener, tophat, stretched, clahe)
    All arrays are same HxW (single channel) and background remains 0.
    """
    # ensure float for computations where needed
    # 1) Grayscale from the foreground RGB (background black -> grayscale 0)
    gray = grayscale_custom(rgb_fg)

    # Force background to 0 explicitly (in case rgb_fg had non-zero due to compression)
    gray = np.where(mask_fg == 1, gray, 0).astype(np.uint8)

    # 2) Wiener filter (small neighborhood)
    denoised = wiener(gray.astype(float), (2, 2))
    denoised = np.clip(denoised, 0, 255).astype(np.uint8)
    denoised = np.where(mask_fg == 1, denoised, 0).astype(np.uint8)

    # 3) Top-hat filtering (illumination correction)
    selem = disk(15)
    tophat = white_tophat(denoised, selem)
    tophat = np.clip(tophat, 0, 255).astype(np.uint8)
    tophat = np.where(mask_fg == 1, tophat, 0).astype(np.uint8)

    # 4) Contrast stretching (percentile-based) -- compute percentiles only on foreground pixels
    fg_pixels = tophat[mask_fg == 1]
    if fg_pixels.size == 0:
        # nothing in foreground; return zeros
        stretched = np.zeros_like(tophat)
    else:
        p_low, p_high = np.percentile(fg_pixels, (15, 90))
        # if p_low == p_high, rescale_intensity would fail; handle it
        if p_low == p_high:
            stretched = tophat.copy()
        else:
            stretched = exposure.rescale_intensity(
                tophat, in_range=(p_low, p_high), out_range=(0, 255)
            ).astype(np.uint8)
    stretched = np.where(mask_fg == 1, stretched, 0).astype(np.uint8)

    # 5) CLAHE (applied only to foreground pixels)
    clahe = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(1, 10))
    final_clahe = clahe.apply(stretched)
    final_clahe = np.where(mask_fg == 1, final_clahe, 0).astype(np.uint8)

    return gray, denoised, tophat, stretched, final_clahe

# ---------------------------
# COLLECT INPUT FILES
# ---------------------------
files = [p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in EXTS]
files = sorted(files)
print(f"Found {len(files)} images in {INPUT_DIR}")

# ---------------------------
# PROCESS ALL IMAGES
# ---------------------------
preview_list = []
for idx, img_path in enumerate(tqdm(files, desc="Processing images")):
    safe_name = safe_name_from_path(img_path, INPUT_DIR)
    stem = Path(safe_name).stem

    # 0) Extract foreground (GrabCut)
    try:
        fg_rgba, mask_fg = extract_foreground_grabcut(img_path, iter_count=5, rect_margin=10)
    except Exception as e:
        print(f"Error extracting foreground for {img_path}: {e}")
        continue

    # save raw fg RGBA
    Image.fromarray(fg_rgba, mode="RGBA").save(STEP0_DIR / f"{stem}_fg_raw.png")

    # prepare rgb foreground (background = black)
    rgb_fg = fg_rgba[..., :3]

    # Run enhancement pipeline on foreground only
    try:
        gray, denoised, tophat, stretched, final_clahe = enhance_pipeline_on_foreground(rgb_fg, mask_fg)
    except Exception as e:
        print(f"Error enhancing foreground for {img_path}: {e}")
        continue

    # Save per-step foreground images (single-channel PNG)
    Image.fromarray(gray).save(STEP1_DIR / f"{stem}_fg_step1_gray.png")
    Image.fromarray(denoised).save(STEP2_DIR / f"{stem}_fg_step2_wiener.png")
    Image.fromarray(tophat).save(STEP3_DIR / f"{stem}_fg_step3_tophat.png")
    Image.fromarray(stretched).save(STEP4_DIR / f"{stem}_fg_step4_stretch.png")
    Image.fromarray(final_clahe).save(STEP5_DIR / f"{stem}_fg_step5_CLAHE.png")

    # Create final RGBA: use processed final_clahe as RGB channels (so the subject shows processed grayscale),
    # preserve the original alpha from GrabCut so the background remains transparent.
    rgb_processed = np.dstack((final_clahe, final_clahe, final_clahe)).astype(np.uint8)
    alpha = fg_rgba[..., 3]
    final_rgba = np.dstack((rgb_processed, alpha))
    Image.fromarray(final_rgba, mode="RGBA").save(STEP_FINAL_DIR / f"{stem}_fg_enhanced.png")

    # Collect previews for the first 5 images
    if idx < 5:
        preview_list.append((img_path.name, fg_rgba, mask_fg * 255, gray, denoised, tophat, stretched, final_clahe, final_rgba))

print("Processing complete.")
print("Saved results to:", OUTPUT_DIR)

# ---------------------------
# SHOW FIRST 5 RESULTS
# ---------------------------
for item in preview_list:
    (name, fg_rgba, mask_vis, gray, denoised, tophat, stretched, final_clahe, final_rgba) = item
    titles = ["Foreground RGBA", "Mask", "Gray", "Wiener", "Top-hat", "Stretch", "CLAHE", "Final RGBA"]
    images = [fg_rgba, mask_vis, gray, denoised, tophat, stretched, final_clahe, final_rgba]

    plt.figure(figsize=(18, 3))
    plt.suptitle(f"Preview: {name}", fontsize=14)

    for j, (title, img) in enumerate(zip(titles, images)):
        plt.subplot(1, len(images), j+1)
        if img is None:
            plt.text(0.5, 0.5, "N/A", ha="center", va="center")
            plt.title(title)
            plt.axis("off")
            continue

        if img.ndim == 2:
            plt.imshow(img, cmap="gray")
        else:
            # RGBA or RGB
            plt.imshow(img)
        plt.title(title)
        plt.axis("off")

    plt.show()


In [ ]:
#METHODS TO BE APPLiED
# METHOD-01 : HONG'S GABOR FILTERING, LINK : https://colab.research.google.com/drive/1SBh0KRkrZ2EWgGbilip9XYHqqyy7mmBE?usp=sharing
# METHOD 02 : STFT fingerprint enhancement method, LINK :
# METHOD-03 : ODF orientation diffusion filtering method, LINK :

In [ ]:
#POSTPROCESSING
# Folder version: adaptive binarization (Sauvola) + skeletonization
# Requires: numpy, opencv-python, scikit-image, matplotlib, tqdm
!pip install -q scikit-image tqdm

import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
from skimage.filters import threshold_sauvola
from skimage.morphology import skeletonize
from tqdm import tqdm
import os

# ---------------- USER SETTINGS ----------------
INPUT_DIR   = Path("/content/enhanced_dataset_all/fingerprint_gabor_batch/gabor_soft")      # folder containing images to binarize (change if needed)
OUTPUT_DIR  = Path("CLAHE_bin_and_skel") # root output folder
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BINARY_DIR   = OUTPUT_DIR / "CLAHE_binary"
SKELETON_DIR = OUTPUT_DIR / "CLAHE_skeleton"
BINARY_DIR.mkdir(exist_ok=True)
SKELETON_DIR.mkdir(exist_ok=True)

EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

# Sauvola params (tweak to taste)
WINDOW_SIZE = 25   # try 15..35
K = 0.2            # Sauvola k parameter

# Preview settings
MAX_PREVIEW = 9    # how many images to show in a grid preview

# ---------------- collect files ----------------
files = sorted([p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in EXTS])
print(f"Found {len(files)} image(s) in {INPUT_DIR}")

if len(files) == 0:
    raise SystemExit("No images found in INPUT_DIR — adjust the path or add images.")

preview = []  # store tuples (title, ink, skel_ink)

# ---------------- process each image ----------------
for p in tqdm(files, desc="Binarizing & skeletonizing"):
    try:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            print("Warning: unable to read", p)
            continue

        # 1) normalize contrast to full 0..255
        img_norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        # 2) Sauvola threshold (skimage expects float in many examples)
        # convert to float in [0,255]
        th = threshold_sauvola(img_norm.astype(np.uint8), window_size=WINDOW_SIZE, k=K)
        binary_bool = img_norm > th               # ridges = True (white)
        binary = (binary_bool.astype(np.uint8) * 255)

        # 3) invert to get ink style: ridges black, background white
        ink = 255 - binary   # uint8, ridges black (0), bg white (255)

        # 4) skeletonization (thin ridges)
        # skeletonize expects boolean where True are foreground pixels.
        # Our ridges are black (0) in ink, so invert them to get foreground True
        ridges_bool = (ink == 0)
        skel_bool = skeletonize(ridges_bool)   # result: True where skeleton present
        skel_img = (skel_bool.astype(np.uint8) * 255)  # skeleton white on black
        skel_ink = 255 - skel_img                # skeleton black on white (ink style)

        # ---------- save outputs ----------
        # create a safe filename preserving relative path
        rel = p.relative_to(INPUT_DIR)
        safe_name = str(rel).replace(os.sep, "__")
        base_stem = Path(safe_name).stem

        bin_out = BINARY_DIR / f"{base_stem}_binary.png"
        skel_out = SKELETON_DIR / f"{base_stem}_skeleton.png"

        cv2.imwrite(str(bin_out), binary)
        cv2.imwrite(str(skel_out), skel_ink)

        # keep for preview
        if len(preview) < MAX_PREVIEW:
            preview.append((p.name, ink, skel_ink))

    except Exception as e:
        print("Error processing", p, ":", e)
        continue

print(f"Done. Saved binaries to: {BINARY_DIR}")
print(f"Done. Saved skeletons to: {SKELETON_DIR}")

# ---------------- show previews ----------------
if len(preview) > 0:
    n = len(preview)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols*4, rows*3))
    for i, (title, ink_img, skel_img) in enumerate(preview):
        plt.subplot(rows, cols*2, 2*i+1)
        plt.imshow(ink_img, cmap="gray")
        plt.title(f"{title}\nInk (thick)")
        plt.axis("off")

        plt.subplot(rows, cols*2, 2*i+2)
        plt.imshow(skel_img, cmap="gray")
        plt.title("Skeleton (thin)")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No preview images to show.")
